# Qwen2.5-1.5B-Instruct gốc — inference và đánh giá metric trên Colab

Notebook này **không fine-tune** model. Mục tiêu là load trực tiếp model gốc `Qwen/Qwen2.5-1.5B-Instruct`, sinh tóm tắt trên validation/test set, tính cùng bộ metric như notebook Qwen đã fine-tune, rồi tạo bảng so sánh với kết quả fine-tuned nếu file metric tồn tại.

Dữ liệu Colab mặc định:

```text
/content/NLP/data/val.parquet
/content/NLP/data/test.parquet    # nếu đã có test set
```

Output của model gốc được lưu riêng tại:

```text
/content/NLP/output/qwen2_5_1_5b_origin_inference_colab
```


## Cell 1 — Cài đặt thư viện

Cell này cài các thư viện cần cho inference và đánh giá: `transformers` để load Qwen, `bitsandbytes` để load 4-bit tiết kiệm VRAM, `datasets` để đọc parquet, `evaluate` để tính ROUGE/BLEU/BERTScore.


In [1]:
!pip install -q -U "transformers<4.40.0" datasets accelerate bitsandbytes evaluate rouge_score sacrebleu bert_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 140.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 44.4 MB/s eta 0:00:00


## Cell 2 — Import thư viện và khai báo cấu hình Colab

Cell này khai báo đường dẫn dữ liệu/output, cấu hình model gốc, cấu hình sinh summary và các metric. Vì đây là notebook baseline nên **không có LoRA, không có QLoRA, không có Trainer và không train**.


In [2]:
import os
import gc
import json
import shutil
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import evaluate
from IPython.display import display
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ============================================================
# 1. Cấu hình chung
# ============================================================
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
SEED = 42

# Bật DEBUG_MODE=True nếu chỉ muốn chạy thử nhanh với ít mẫu.
DEBUG_MODE = False
DEBUG_VAL_SIZE = 20
DEBUG_TEST_SIZE = 20

# ============================================================
# 2. Đường dẫn Colab
# ============================================================
PROJECT_DIR = Path("/content/NLP")
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_ROOT = PROJECT_DIR / "output"

# Notebook này chỉ cần validation/test để đánh giá model gốc, không cần train.parquet.
VAL_CANDIDATES = [
    DATA_DIR / "val.parquet",
    DATA_DIR / "valid.parquet",
    DATA_DIR / "validation.parquet",
]

TEST_CANDIDATES = [
    DATA_DIR / "test.parquet",
    DATA_DIR / "private_test.parquet",
]

# Output riêng cho model gốc chưa fine-tune.
RUN_NAME = "qwen2_5_1_5b_origin_inference_colab"
OUTPUT_DIR = OUTPUT_ROOT / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VAL_PRED_CSV = OUTPUT_DIR / "qwen_origin_validation_predictions.csv"
VAL_METRICS_JSON = OUTPUT_DIR / "qwen_origin_validation_metrics.json"

TEST_PRED_CSV = OUTPUT_DIR / "qwen_origin_test_predictions.csv"
TEST_METRICS_JSON = OUTPUT_DIR / "qwen_origin_test_metrics.json"

COMPARE_VAL_CSV = OUTPUT_DIR / "qwen_origin_vs_finetuned_validation_comparison.csv"
COMPARE_TEST_CSV = OUTPUT_DIR / "qwen_origin_vs_finetuned_test_comparison.csv"
OUTPUT_ZIP = OUTPUT_ROOT / f"{RUN_NAME}_outputs.zip"

# Các đường dẫn metric của model đã fine-tune. Notebook sẽ tự dùng file đầu tiên tìm thấy.
FINETUNED_VAL_METRICS_CANDIDATES = [
    OUTPUT_ROOT / "qwen2_5_1_5b_qlora_a100_40gb" / "qwen_a100_40gb_validation_metrics.json",
    OUTPUT_ROOT / "qwen2_5_1_5b_qlora_colab" / "qwen_validation_metrics.json",
    OUTPUT_ROOT / "qwen2_5_1_5b_qlora_colab" / "qwen_a100_40gb_validation_metrics.json",
]

FINETUNED_TEST_METRICS_CANDIDATES = [
    OUTPUT_ROOT / "qwen2_5_1_5b_qlora_a100_40gb" / "qwen_a100_40gb_test_metrics.json",
    OUTPUT_ROOT / "qwen2_5_1_5b_qlora_colab" / "qwen_test_metrics.json",
    OUTPUT_ROOT / "qwen2_5_1_5b_qlora_colab" / "qwen_a100_40gb_test_metrics.json",
]

# ============================================================
# 3. Cấu hình inference cho A100 40GB
# ============================================================
# MAX_PROMPT_LENGTH là độ dài tối đa phần prompt + article đưa vào model.
MAX_PROMPT_LENGTH = 768

# Số token mới tối đa model được sinh cho summary.
MAX_NEW_TOKENS = 128

# A100 40GB thường chạy được batch 8 với Qwen2.5-1.5B 4-bit.
# Nếu bị OOM, giảm GENERATION_BATCH_SIZE xuống 4 hoặc 2.
GENERATION_BATCH_SIZE = 4

# Baseline nên dùng decoding ổn định để so sánh công bằng.
NUM_BEAMS = 1
DO_SAMPLE = False
TEMPERATURE = 1.0
TOP_P = 1.0
REPETITION_PENALTY = 1.1
NO_REPEAT_NGRAM_SIZE = 3

# ============================================================
# 4. Cấu hình metric
# ============================================================
RUN_BERTSCORE = True
BERTSCORE_LANG = "vi"
METRIC_BATCH_SIZE = 8
TOO_SHORT_MIN_TOKENS = 8
TOO_SHORT_RATIO = 0.35

# ============================================================
# 5. Seed và precision
# ============================================================
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

SUPPORTS_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if SUPPORTS_BF16 else torch.float16

print("===== QWEN ORIGIN INFERENCE CONFIG =====")
print(f"MODEL_NAME: {MODEL_NAME}")
print(f"PROJECT_DIR: {PROJECT_DIR}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"DEBUG_MODE: {DEBUG_MODE}")
print(f"MAX_PROMPT_LENGTH: {MAX_PROMPT_LENGTH}")
print(f"MAX_NEW_TOKENS: {MAX_NEW_TOKENS}")
print(f"GENERATION_BATCH_SIZE: {GENERATION_BATCH_SIZE}")
print(f"RUN_BERTSCORE: {RUN_BERTSCORE}")
print(f"COMPUTE_DTYPE: {COMPUTE_DTYPE}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"Total GPU VRAM: {total_vram:.1f} GB")


===== QWEN ORIGIN INFERENCE CONFIG =====
MODEL_NAME: Qwen/Qwen2.5-1.5B-Instruct
PROJECT_DIR: /content/NLP
DATA_DIR: /content/NLP/data
OUTPUT_DIR: /content/NLP/output/qwen2_5_1_5b_origin_inference_colab
DEBUG_MODE: False
MAX_PROMPT_LENGTH: 768
MAX_NEW_TOKENS: 128
GENERATION_BATCH_SIZE: 4
RUN_BERTSCORE: True
COMPUTE_DTYPE: torch.bfloat16
CUDA available: True
GPU: NVIDIA L4
Total GPU VRAM: 22.0 GB


## Cell 3 — Kiểm tra GPU và dọn bộ nhớ

Cell này kiểm tra GPU Colab bằng `nvidia-smi`, sau đó dọn RAM/VRAM trước khi load model. Nếu đang dùng A100 40GB, notebook có thể chạy inference batch lớn hơn T4.


In [3]:
!nvidia-smi

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Wed Jun 10 16:34:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   37C    P8             13W /   72W |       3MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Cell 4 — Hàm tìm file dữ liệu trong thư mục Colab

Cell này tự tìm `val.parquet` hoặc các tên validation tương đương. Test set là tùy chọn, vì có thể vài hôm sau mới được cung cấp.


In [4]:
def first_existing_path(candidates, required=True, name="file"):
    """Trả về đường dẫn đầu tiên tồn tại trong danh sách candidates."""
    for path in candidates:
        path = Path(path)
        if path.exists():
            return path

    if required:
        checked = "\n".join(str(Path(p)) for p in candidates)
        raise FileNotFoundError(
            f"Không tìm thấy {name}. Các đường dẫn đã kiểm tra:\n{checked}"
        )

    return None


VAL_PATH = first_existing_path(VAL_CANDIDATES, required=True, name="validation file")
TEST_PATH = first_existing_path(TEST_CANDIDATES, required=False, name="test file")

print(f"VAL_PATH:  {VAL_PATH}")
print(f"TEST_PATH: {TEST_PATH if TEST_PATH is not None else 'Chưa có test file'}")


VAL_PATH:  /content/NLP/data/val.parquet
TEST_PATH: /content/NLP/data/test.parquet


## Cell 5 — Load tokenizer và validation set

Cell này load tokenizer của Qwen và đọc validation set từ parquet. Dataset phải có hai cột `article` và `summary`. Notebook baseline không đọc `train.parquet` vì không fine-tune.


In [5]:
print("Đang tải tokenizer của Qwen...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Với decoder-only model như Qwen, padding bên trái phù hợp hơn khi generate theo batch.
tokenizer.padding_side = "left"

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Đang tải validation parquet...")
dataset = load_dataset(
    "parquet",
    data_files={"validation": str(VAL_PATH)},
)

required_columns = {"article", "summary"}
missing = required_columns - set(dataset["validation"].column_names)
if missing:
    raise ValueError(
        f"Validation set thiếu các cột bắt buộc: {missing}. "
        f"Các cột hiện có: {dataset['validation'].column_names}"
    )

if DEBUG_MODE:
    val_n = min(DEBUG_VAL_SIZE, len(dataset["validation"]))
    dataset["validation"] = dataset["validation"].select(range(val_n))
    print(f"DEBUG_MODE=True → validation={val_n}")

print(dataset)
print("Ví dụ một mẫu validation:")
print(dataset["validation"][0])


Đang tải tokenizer của Qwen...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Đang tải validation parquet...


Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    validation: Dataset({
        features: ['article', 'summary'],
        num_rows: 1349
    })
})
Ví dụ một mẫu validation:
{'article': "Giải thưởng công bố gần đây bởi World Travel Awards. Đây là năm thứ hai liên tiếp InterContinental Phu Quoc Long Beach Resort được vinh danh ở hạng mục gia đình trên toàn châu Á. Khu nghỉ dưỡng tọa lạc bên biển Phú Quốc, nổi bật với thiết kế lấy cảm hứng từ đại dương. Khuôn viên rộng rãi với nhiều mảng xanh thiên nhiên đậm chất nhiệt đới. Không gian sảnh lễ tân, phòng nghỉ, villa, nhà hàng... đều được chú trọng để tạo sự hài hòa với biển và cây cối. Du khách có thể chọn nghỉ ngơi tại khu phòng khách sạn rộng rãi, tiện nghi, hoặc những căn hộ, phòng suite, biệt thự cao cấp hướng biển. Mỗi không gian được thiết kế dựa trên tinh thần gắn kết các thành viên trong gia đình. Bên cạnh tiện nghi sang trọng, InterContinental Phu Quoc còn hút khách gia đình nhờ loạt trải nghiệm giải trí, thư giãn đa dạng, phù hợp với mọi độ tuổi. Với trẻ nhỏ, k

## Cell 6 — Hàm làm sạch văn bản

Cell này định nghĩa hàm chuẩn hóa văn bản cơ bản để tránh lỗi `None`, `NaN` hoặc khoảng trắng thừa khi generate và tính metric.


In [6]:
def clean_text(text):
    """Chuẩn hóa text cơ bản để tránh lỗi None, NaN hoặc khoảng trắng thừa."""
    if text is None:
        return ""
    text = str(text)
    text = " ".join(text.split())
    return text.strip()


## Cell 7 — Load trực tiếp model gốc Qwen ở chế độ 4-bit

Cell này load **model gốc chưa fine-tune**. Không gắn LoRA adapter, không gọi `SFTTrainer`, không train. 4-bit chỉ dùng để tiết kiệm VRAM khi inference trên Colab, không làm thay đổi trọng số gốc trên ổ đĩa.


In [7]:
print("Đang tải model gốc Qwen ở chế độ 4-bit để inference...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model.config.use_cache = True
model.eval()

print("Đã load model gốc chưa fine-tune.")
print(model.__class__.__name__)


Đang tải model gốc Qwen ở chế độ 4-bit để inference...


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Đã load model gốc chưa fine-tune.
Qwen2ForCausalLM


## Cell 8 — Hàm tạo prompt và sinh tóm tắt

Cell này tạo prompt ChatML giống notebook fine-tuned để so sánh công bằng hơn. Sau đó định nghĩa hàm generate summary theo batch.


In [8]:
def build_inference_prompt(article):
    """Tạo prompt ChatML cho bài toán tóm tắt."""
    article = clean_text(article)
    messages = [
        {
            "role": "system",
            "content": "Bạn là hệ thống tóm tắt văn bản tiếng Việt. Hãy tóm tắt ngắn gọn, đúng ý chính, không bịa thông tin.",
        },
        {
            "role": "user",
            "content": f"Tóm tắt văn bản sau bằng tiếng Việt:\n\n{article}",
        },
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def postprocess_summary(text):
    """Làm sạch output sinh ra từ model."""
    text = clean_text(text)
    text = text.replace("<|im_end|>", "").replace("<|im_start|>", "")
    return text.strip()


def generate_summaries(articles, batch_size=GENERATION_BATCH_SIZE):
    """Sinh summary cho danh sách article bằng model gốc."""
    predictions = []
    model.eval()

    for start in tqdm(range(0, len(articles), batch_size), desc="Generating summaries"):
        batch_articles = articles[start:start + batch_size]
        prompts = [build_inference_prompt(article) for article in batch_articles]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_PROMPT_LENGTH,
        )
        inputs = {key: value.to(model.device) for key, value in inputs.items()}
        prompt_token_length = inputs["input_ids"].shape[1]

        generate_kwargs = dict(
            max_new_tokens=MAX_NEW_TOKENS,
            num_beams=NUM_BEAMS,
            do_sample=DO_SAMPLE,
            repetition_penalty=REPETITION_PENALTY,
            no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        if DO_SAMPLE:
            generate_kwargs["temperature"] = TEMPERATURE
            generate_kwargs["top_p"] = TOP_P

        with torch.no_grad():
            outputs = model.generate(**inputs, **generate_kwargs)

        generated_tokens = outputs[:, prompt_token_length:]
        decoded = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        predictions.extend([postprocess_summary(item) for item in decoded])

        del inputs, outputs, generated_tokens
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return predictions


## Cell 9 — Chạy thử inference trên một mẫu validation

Cell này sinh thử một prediction để kiểm tra model gốc có chạy đúng pipeline không trước khi generate toàn bộ validation set.


In [9]:
sample = dataset["validation"][0]
print("===== ARTICLE =====")
print(clean_text(sample["article"])[:2000])
print("\n===== REFERENCE SUMMARY =====")
print(clean_text(sample["summary"]))
print("\n===== ORIGIN MODEL PREDICTION =====")
print(generate_summaries([sample["article"]], batch_size=1)[0])


===== ARTICLE =====
Giải thưởng công bố gần đây bởi World Travel Awards. Đây là năm thứ hai liên tiếp InterContinental Phu Quoc Long Beach Resort được vinh danh ở hạng mục gia đình trên toàn châu Á. Khu nghỉ dưỡng tọa lạc bên biển Phú Quốc, nổi bật với thiết kế lấy cảm hứng từ đại dương. Khuôn viên rộng rãi với nhiều mảng xanh thiên nhiên đậm chất nhiệt đới. Không gian sảnh lễ tân, phòng nghỉ, villa, nhà hàng... đều được chú trọng để tạo sự hài hòa với biển và cây cối. Du khách có thể chọn nghỉ ngơi tại khu phòng khách sạn rộng rãi, tiện nghi, hoặc những căn hộ, phòng suite, biệt thự cao cấp hướng biển. Mỗi không gian được thiết kế dựa trên tinh thần gắn kết các thành viên trong gia đình. Bên cạnh tiện nghi sang trọng, InterContinental Phu Quoc còn hút khách gia đình nhờ loạt trải nghiệm giải trí, thư giãn đa dạng, phù hợp với mọi độ tuổi. Với trẻ nhỏ, khu Planet Trekker là nơi các bé có thể thoải mái vui chơi, học hỏi từ các đầu sách thiếu nhi, những buổi workshop thủ công... Phụ huyn

Generating summaries:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


, martini hay các loại thức uống đặc trưng của địa phương. Resort cũng tổ chức các chương trình văn nghệ, biểu diễn âm nhạc, hội chợ du lịch, sự kiện giải trí ngoài trời... nhằm tạo ra không giao tiếp thân thiện, hấp dẫn du khách. Resort sở hữu đội ngũ nhân viên chuyên nghiệp, tận tụy, luôn sẵn lòng hỗ trợ du khách trong mọi tình huống.


## Cell 10 — Sinh prediction trên validation set

Cell này generate summary cho toàn bộ validation set và lưu ra CSV. File CSV này dùng để đọc ví dụ định tính và tính metric định lượng.


In [10]:
val_df = pd.DataFrame(dataset["validation"])

if DEBUG_MODE:
    val_df = val_df.head(DEBUG_VAL_SIZE).copy()

val_articles = val_df["article"].astype(str).tolist()
val_references = val_df["summary"].astype(str).tolist()

val_predictions = generate_summaries(val_articles)

val_result_df = pd.DataFrame({
    "article": val_articles,
    "reference": val_references,
    "prediction": val_predictions,
})

val_result_df.to_csv(VAL_PRED_CSV, index=False, encoding="utf-8-sig")
print(f"Đã lưu prediction validation tại: {VAL_PRED_CSV}")
val_result_df.head()


Generating summaries:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Đã lưu prediction validation tại: /content/NLP/output/qwen2_5_1_5b_origin_inference_colab/qwen_origin_validation_predictions.csv


,article,reference,prediction
0,Giải thưởng công bố gần đây bởi World Travel A...,InterContinental Phu Quoc Long Beach Resort đã...,", martini, rượu vèn... tại Bar Viva. Resort cũ..."
1,Theo bảng xếp hạng 20 quốc gia tốt nhất thế gi...,Việt Nam đã xếp hạng 15 trên bảng xếp hạng 20 ...,Dựa trên bảng xếp loại năm ước mơ du lịch của ...
2,"Ngày hội Văn hóa, Thể thao và Du lịch các dân ...","Ngày hội Văn hóa, Thể thao và Du lịch các dân ...","Ngày Hội Văn Hóa, Thể Thao Và Du Lịch Các Dân ..."
3,Giải thưởng do Tạp chí du lịch Condé Nast Trav...,"Phú Quốc, đảo ngọc của Việt Nam, đã được vinh ...",du khách tới từ 160 quốc gia và vùng lãnh thổ.
4,KKday Vietnam vừa công bố hợp tác chiến lược S...,KKday Vietnam vừa công bố hợp tác chiến lược v...,KKday Việt Nam đã ký hợp tác战略合作 với Swiss Tra...


## Cell 11 — Hàm tính ROUGE, BERTScore, BLEU và các metric độ dài

Cell này dùng cùng bộ metric với notebook fine-tuned: ROUGE, BLEU, BERTScore, tỷ lệ prediction rỗng, tỷ lệ quá ngắn và thống kê độ dài. Các metric này giúp so sánh model gốc với model đã fine-tune.


In [16]:
def token_count(text):
    """Đếm token đơn giản theo khoảng trắng để tính thống kê độ dài."""
    return len(clean_text(text).split())


def safe_mean(values):
    return float(np.mean(values)) if len(values) > 0 else 0.0


def compute_summarization_metrics(predictions, references, output_json_path=None):
    """Tính các metric chính cho bài toán tóm tắt."""

    predictions = [clean_text(item) for item in predictions]
    references = [clean_text(item) for item in references]

    rouge_metric = evaluate.load("rouge")
    sacrebleu_metric = evaluate.load("sacrebleu")

    rouge_results = rouge_metric.compute(
        predictions=predictions,
        references=references,
        use_stemmer=False,
    )

    bleu_results = sacrebleu_metric.compute(
        predictions=predictions,
        references=[[ref] for ref in references],
    )

    pred_lengths = [token_count(item) for item in predictions]
    ref_lengths = [token_count(item) for item in references]

    length_ratios = [
        pred_len / ref_len if ref_len > 0 else 0.0
        for pred_len, ref_len in zip(pred_lengths, ref_lengths)
    ]

    empty_prediction_rate = safe_mean([1.0 if pred_len == 0 else 0.0 for pred_len in pred_lengths])

    too_short_flags = []
    for pred_len, ref_len, ratio in zip(pred_lengths, ref_lengths, length_ratios):
        is_too_short = (pred_len < TOO_SHORT_MIN_TOKENS) or (ref_len > 0 and ratio < TOO_SHORT_RATIO)
        too_short_flags.append(1.0 if is_too_short else 0.0)

    metrics = {
        "num_samples": len(predictions),
        "rouge1": float(rouge_results["rouge1"]),
        "rouge2": float(rouge_results["rouge2"]),
        "rougeL": float(rouge_results["rougeL"]),
        "rougeLsum": float(rouge_results["rougeLsum"]),
        "bleu": float(bleu_results["score"]),
        "empty_prediction_rate": float(empty_prediction_rate),
        "too_short_rate": float(safe_mean(too_short_flags)),
        "avg_prediction_tokens": float(safe_mean(pred_lengths)),
        "avg_reference_tokens": float(safe_mean(ref_lengths)),
        "avg_length_ratio": float(safe_mean(length_ratios)),
    }

    if RUN_BERTSCORE:
        pass

    if output_json_path is not None:
        output_json_path = Path(output_json_path)
        output_json_path.parent.mkdir(parents=True, exist_ok=True)
        with open(output_json_path, "w", encoding="utf-8") as f:
            json.dump(metrics, f, ensure_ascii=False, indent=2)

    return metrics


def show_metrics(metrics):
    display(pd.DataFrame([metrics]).T.rename(columns={0: "value"}))


## Cell 12 — Tính metric trên validation set

Cell này tính metric cho prediction validation của model gốc và lưu ra JSON. Đây là file chính để so sánh với notebook Qwen fine-tuned.


In [17]:
val_metrics = compute_summarization_metrics(
    predictions=val_result_df["prediction"].tolist(),
    references=val_result_df["reference"].tolist(),
    output_json_path=VAL_METRICS_JSON,
)

print("===== ORIGIN MODEL VALIDATION METRICS =====")
show_metrics(val_metrics)
print(f"Đã lưu metric validation của model gốc tại: {VAL_METRICS_JSON}")


===== ORIGIN MODEL VALIDATION METRICS =====


,value
num_samples,1349.000000
rouge1,0.565730
rouge2,0.228424
rougeL,0.311733
rougeLsum,0.311722
bleu,2.830031
empty_prediction_rate,0.000741
too_short_rate,0.132691
avg_prediction_tokens,72.306894
avg_reference_tokens,102.068199


Đã lưu metric validation của model gốc tại: /content/NLP/output/qwen2_5_1_5b_origin_inference_colab/qwen_origin_validation_metrics.json


## Cell 13 — So sánh validation metric giữa model gốc và model fine-tuned

Cell này tự tìm file metric của notebook fine-tuned. Nếu tìm thấy, nó tạo bảng so sánh `origin`, `fine_tuned` và `fine_tuned_minus_origin`. Nếu chưa có file fine-tuned metric, hãy chạy notebook fine-tuned trước hoặc copy file JSON vào đúng thư mục output.


In [ ]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def first_existing_optional(candidates):
    for path in candidates:
        path = Path(path)
        if path.exists():
            return path
    return None


def compare_metrics(origin_metrics, finetuned_metrics, output_csv_path=None):
    all_keys = sorted(set(origin_metrics.keys()) | set(finetuned_metrics.keys()))
    rows = []
    for key in all_keys:
        origin_value = origin_metrics.get(key, np.nan)
        finetuned_value = finetuned_metrics.get(key, np.nan)

        if isinstance(origin_value, (int, float)) and isinstance(finetuned_value, (int, float)):
            diff = finetuned_value - origin_value
        else:
            diff = np.nan

        rows.append({
            "metric": key,
            "origin": origin_value,
            "fine_tuned": finetuned_value,
            "fine_tuned_minus_origin": diff,
        })

    compare_df = pd.DataFrame(rows)
    if output_csv_path is not None:
        compare_df.to_csv(output_csv_path, index=False, encoding="utf-8-sig")
    return compare_df


finetuned_val_metrics_path = first_existing_optional(FINETUNED_VAL_METRICS_CANDIDATES)

if finetuned_val_metrics_path is None:
    print("Chưa tìm thấy file validation metric của model fine-tuned.")
    print("Notebook đã kiểm tra các đường dẫn:")
    for path in FINETUNED_VAL_METRICS_CANDIDATES:
        print(f"- {path}")
else:
    finetuned_val_metrics = load_json(finetuned_val_metrics_path)
    comparison_df = compare_metrics(
        origin_metrics=val_metrics,
        finetuned_metrics=finetuned_val_metrics,
        output_csv_path=COMPARE_VAL_CSV,
    )
    print(f"Đã tìm thấy metric fine-tuned tại: {finetuned_val_metrics_path}")
    print(f"Đã lưu bảng so sánh validation tại: {COMPARE_VAL_CSV}")
    display(comparison_df)


## Cell 14 — Module test cho `test.parquet`

Cell này dùng khi bạn đã có test set thật. Đặt file tại `/content/NLP/data/test.parquet`, sau đó chạy cell này để sinh prediction và tính metric test cho model gốc.


In [18]:
TEST_PATH = first_existing_path(TEST_CANDIDATES, required=False, name="test file")

if TEST_PATH is None:
    print("Chưa tìm thấy test file.")
    print("Khi có test set, hãy đặt file tại một trong các đường dẫn sau:")
    for path in TEST_CANDIDATES:
        print(f"- {path}")
else:
    print(f"Đang tải test set từ: {TEST_PATH}")

    test_dataset = load_dataset(
        "parquet",
        data_files={"test": str(TEST_PATH)},
    )["test"]

    missing = required_columns - set(test_dataset.column_names)
    if missing:
        raise ValueError(
            f"Test set thiếu các cột bắt buộc: {missing}. "
            f"Các cột hiện có: {test_dataset.column_names}"
        )

    if DEBUG_MODE:
        test_n = min(DEBUG_TEST_SIZE, len(test_dataset))
        test_dataset = test_dataset.select(range(test_n))
        print(f"DEBUG_MODE=True → test={test_n}")

    test_df = pd.DataFrame(test_dataset)
    test_articles = test_df["article"].astype(str).tolist()
    test_references = test_df["summary"].astype(str).tolist()

    test_predictions = generate_summaries(test_articles)

    test_result_df = pd.DataFrame({
        "article": test_articles,
        "reference": test_references,
        "prediction": test_predictions,
    })

    test_result_df.to_csv(TEST_PRED_CSV, index=False, encoding="utf-8-sig")
    print(f"Đã lưu prediction test tại: {TEST_PRED_CSV}")

    test_metrics = compute_summarization_metrics(
        predictions=test_result_df["prediction"].tolist(),
        references=test_result_df["reference"].tolist(),
        output_json_path=TEST_METRICS_JSON,
    )

    print("===== ORIGIN MODEL TEST METRICS =====")
    show_metrics(test_metrics)
    print(f"Đã lưu metric test của model gốc tại: {TEST_METRICS_JSON}")


Đang tải test set từ: /content/NLP/data/test.parquet


Generating test split: 0 examples [00:00, ? examples/s]

Generating summaries:   0%|          | 0/336 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Cell 15 — So sánh test metric giữa model gốc và model fine-tuned

Cell này chỉ chạy được khi cả model gốc và model fine-tuned đều đã có file test metric. Nếu chưa có test set hoặc chưa chạy notebook fine-tuned trên test set, cell sẽ in hướng dẫn thay vì báo lỗi.


In [ ]:
origin_test_metrics_path = TEST_METRICS_JSON if TEST_METRICS_JSON.exists() else None
finetuned_test_metrics_path = first_existing_optional(FINETUNED_TEST_METRICS_CANDIDATES)

if origin_test_metrics_path is None:
    print("Chưa có test metric của model gốc. Hãy chạy Cell 14 sau khi có test.parquet.")
elif finetuned_test_metrics_path is None:
    print("Chưa tìm thấy test metric của model fine-tuned.")
    print("Notebook đã kiểm tra các đường dẫn:")
    for path in FINETUNED_TEST_METRICS_CANDIDATES:
        print(f"- {path}")
else:
    origin_test_metrics = load_json(origin_test_metrics_path)
    finetuned_test_metrics = load_json(finetuned_test_metrics_path)
    test_comparison_df = compare_metrics(
        origin_metrics=origin_test_metrics,
        finetuned_metrics=finetuned_test_metrics,
        output_csv_path=COMPARE_TEST_CSV,
    )
    print(f"Đã tìm thấy test metric fine-tuned tại: {finetuned_test_metrics_path}")
    print(f"Đã lưu bảng so sánh test tại: {COMPARE_TEST_CSV}")
    display(test_comparison_df)


## Cell 16 — Nén output để tải về hoặc lưu Google Drive

Cell này nén prediction CSV, metric JSON và bảng so sánh thành một file zip. Hãy tải file zip về máy hoặc copy sang Google Drive để không mất kết quả khi Colab tắt runtime.


In [19]:
if OUTPUT_ZIP.exists():
    OUTPUT_ZIP.unlink()

shutil.make_archive(
    base_name=str(OUTPUT_ZIP).replace(".zip", ""),
    format="zip",
    root_dir=str(OUTPUT_DIR),
)

print(f"Đã nén output tại: {OUTPUT_ZIP}")

# Nếu muốn copy sang Google Drive, bỏ comment các dòng dưới:
# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE_BACKUP_DIR = Path('/content/drive/MyDrive/NLP/qwen_origin_baseline')
# DRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)
# shutil.copy2(OUTPUT_ZIP, DRIVE_BACKUP_DIR / OUTPUT_ZIP.name)
# print(f"Đã backup zip sang Drive: {DRIVE_BACKUP_DIR / OUTPUT_ZIP.name}")


Đã nén output tại: /content/NLP/output/qwen2_5_1_5b_origin_inference_colab_outputs.zip
